In [14]:
%load_ext autoreload
%autoreload 2

In [8]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

In [9]:
list_of_testing_videos = ["00170.mp4",
"00208.mp4",
"00063.mp4",
"00024.mp4",
"00021.mp4",
"00036.mp4",
"00202.mp4",
"00236.mp4",
"00197.mp4",
"00133.mp4",
"00213.mp4",
"00011.mp4",
"00095.mp4",
"00138.mp4",
"00106.mp4",
"00194.mp4",
"00092.mp4",
"00227.mp4",
"00244.mp4",
"00119.mp4",
"00082.mp4",
"00188.mp4",
"00076.mp4",
"00047.mp4",
"00168.mp4",
"00207.mp4",
"00193.mp4",
"00061.mp4",
"00023.mp4",
"00048.mp4",
"id1_0007.mp4",
"id2_0008.mp4",
"id3_0001.mp4",
"id6_0005.mp4",
"id9_0000.mp4",
"id11_0008.mp4",
"id16_0011.mp4",
"id17_0000.mp4",
"id1_id0_0007.mp4",
"id1_id2_0007.mp4",
"id1_id3_0007.mp4",
"id1_id4_0007.mp4",
"id1_id6_0007.mp4",
"id1_id9_0007.mp4",
"id1_id16_0007.mp4",
"id1_id17_0007.mp4",
"id2_id0_0008.mp4",
"id2_id1_0008.mp4",
"id2_id3_0008.mp4",
"id2_id9_0008.mp4",
"id2_id16_0008.mp4",
"id2_id17_0008.mp4",
"id3_id1_0001.mp4",
"id3_id2_0001.mp4",
"id3_id4_0001.mp4",
"id3_id6_0001.mp4",
"id3_id9_0001.mp4",
"id3_id16_0001.mp4",
"id3_id17_0001.mp4",
"id4_id1_0003.mp4",
"id6_id0_0005.mp4",
"id6_id1_0005.mp4",
"id6_id2_0005.mp4",
"id6_id3_0005.mp4",
"id6_id9_0005.mp4",
"id6_id16_0005.mp4",
"id6_id17_0005.mp4",
"id7_id10_0009.mp4",
"id7_id11_0009.mp4",
"id7_id13_0009.mp4",
"id9_id0_0000.mp4",
"id9_id1_0000.mp4",
"id9_id2_0000.mp4",
"id9_id3_0000.mp4",
"id9_id6_0000.mp4",
"id9_id16_0000.mp4",
"id9_id17_0000.mp4",
"id10_id7_0004.mp4",
"id10_id11_0004.mp4",
"id10_id12_0004.mp4",
"id10_id13_0004.mp4",
"id10_id7_0001.mp4",
"id10_id11_0001.mp4",
"id10_id12_0001.mp4",
"id10_id13_0001.mp4",
"id11_id7_0008.mp4",
"id16_id0_0011.mp4",
"id16_id1_0011.mp4",
"id16_id2_0011.mp4",
"id16_id3_0011.mp4",
"id16_id6_0011.mp4",
"id16_id9_0011.mp4",
"id16_id17_0011.mp4",
"id17_id0_0000.mp4",
"id17_id1_0000.mp4",
"id17_id2_0000.mp4",
"id17_id3_0000.mp4",
"id17_id6_0000.mp4",
"id17_id9_0000.mp4",
"id17_id16_0000.mp4"]

In [10]:

os.listdir('/Users/kunzuhaib/work/student/trimester_02/project/data')


['.DS_Store',
 'Celeb-synthesis',
 'List_of_testing_videos.txt',
 'YouTube-real',
 'Celeb-real']

In [11]:
BASE_PATH = "/Users/kunzuhaib/work/student/trimester_02/project/data"
official_video_split = ['Celeb-synthesis', 'YouTube-real', 'Celeb-real']

split_metadata = []

for dir in official_video_split:
    for video_id in os.listdir(Path(BASE_PATH).joinpath(dir)):
        row = dict(
            video_path=dir + '/' + video_id,
            video_id=video_id,
            source=dir,
            label='real' if 'real' in dir else 'fake',
            is_official_test=1 if video_id in set(list_of_testing_videos) else 0
        )
        split_metadata.append(row)

metadata_df = pd.DataFrame(split_metadata)
metadata_df


,video_path,video_id,source,label,is_official_test
0,Celeb-synthesis/id9_id17_0009.mp4,id9_id17_0009.mp4,Celeb-synthesis,fake,0
1,Celeb-synthesis/id9_id16_0009.mp4,id9_id16_0009.mp4,Celeb-synthesis,fake,0
2,Celeb-synthesis/id3_id0_0000.mp4,id3_id0_0000.mp4,Celeb-synthesis,fake,0
3,Celeb-synthesis/id0_id4_0002.mp4,id0_id4_0002.mp4,Celeb-synthesis,fake,0
4,Celeb-synthesis/id16_id17_0007.mp4,id16_id17_0007.mp4,Celeb-synthesis,fake,0
...,...,...,...,...,...
1198,Celeb-real/id17_0001.mp4,id17_0001.mp4,Celeb-real,real,0
1199,Celeb-real/id0_0006.mp4,id0_0006.mp4,Celeb-real,real,0
1200,Celeb-real/id17_0000.mp4,id17_0000.mp4,Celeb-real,real,1
1201,Celeb-real/id1_0006.mp4,id1_0006.mp4,Celeb-real,real,0


In [15]:
from sklearn.model_selection import train_test_split

test_df = metadata_df[metadata_df['is_official_test'] == 1]
train_pool = metadata_df[metadata_df['is_official_test'] == 0]
train_pool["strata"] = metadata_df["source"] + "_" + metadata_df["label"]

train_df, val_df = train_test_split(
    train_pool,
    test_size=0.20,
    stratify=train_pool["strata"],
    random_state=42
)


/var/folders/2k/7r4vrq9547v2qk888ymft_1h0000gn/T/ipykernel_49884/3859151703.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_pool["strata"] = metadata_df["source"] + "_" + metadata_df["label"]


In [17]:
test_df['split'] = 'test'
train_df['split'] = 'train'
val_df['split'] = 'val'

/var/folders/2k/7r4vrq9547v2qk888ymft_1h0000gn/T/ipykernel_49884/2368505419.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['split'] = 'test'


In [6]:
metadata_df_final_v1 = pd.concat([train_df, val_df, test_df], ignore_index=True)


In [ ]:
test_videos_set = set(metadata_df_final_v1[metadata_df_final_v1['split'] == 'test']['video_id'])
val_videos_set = set(metadata_df_final_v1[metadata_df_final_v1['split'] == 'val']['video_id'])

FACE_CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(FACE_CASCADE_PATH)


def get_output_folder(video_name: str, label: str, base_out_dir: str) -> str:
    if video_name in test_videos_set:
        return os.path.join(base_out_dir, 'test', label)
    elif video_name in val_videos_set:
        return os.path.join(base_out_dir, 'val', label)
    else:
        return os.path.join(base_out_dir, 'train', label)


def crop_largest_face(
    frame: np.ndarray,
    scale_factor: float = 1.1,
    min_neighbors: int = 5,
    min_size: tuple = (60, 60),
    padding_ratio: float = 0.25
):
    """
    Detect the largest face and return a padded crop.
    Returns None if no face is found.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=scale_factor,
        minNeighbors=min_neighbors,
        minSize=min_size
    )

    if len(faces) == 0:
        return None

    x, y, w, h = max(faces, key=lambda b: b[2] * b[3])

    pad_w = int(w * padding_ratio)
    pad_h = int(h * padding_ratio)

    x1 = max(0, x - pad_w)
    y1 = max(0, y - pad_h)
    x2 = min(frame.shape[1], x + w + pad_w)
    y2 = min(frame.shape[0], y + h + pad_h)

    face_crop = frame[y1:y2, x1:x2]
    if face_crop.size == 0:
        return None

    return face_crop


def is_near_duplicate(
    prev_crop_gray: np.ndarray,
    curr_crop_gray: np.ndarray,
    diff_threshold: int = 12,
    change_ratio_threshold: float = 0.015
) -> bool:
    """
    Returns True if crops are too similar.
    """
    if prev_crop_gray is None:
        return False

    target_size = (128, 128)
    a = cv2.resize(prev_crop_gray, target_size)
    b = cv2.resize(curr_crop_gray, target_size)

    diff = cv2.absdiff(a, b)
    _, thresh = cv2.threshold(diff, diff_threshold, 255, cv2.THRESH_BINARY)
    change_ratio = np.count_nonzero(thresh) / thresh.size

    return change_ratio < change_ratio_threshold


def extract_xception_frames(
    source_video_path: str,
    video_name: str,
    output_folder: str,
    frame_interval: int = 10,
    max_frames_per_video: int = 15,
    output_size: tuple = (299, 299),
    duplicate_filter: bool = True,
    diff_threshold: int = 12,
    change_ratio_threshold: float = 0.015,
    save_center_crop_if_no_face: bool = False
) -> list:
    """
    Extract face crops suitable for Xception training.

    Strategy:
    - sample every Nth frame
    - detect largest face
    - skip near-duplicate face crops
    - cap number of saved crops per video
    - resize to Xception size (299x299)
    """
    os.makedirs(output_folder, exist_ok=True)
    saved_paths = []

    cap = cv2.VideoCapture(source_video_path)
    if not cap.isOpened():
        print(f"Could not open video: {source_video_path}")
        return saved_paths

    frame_count = 0
    saved_count = 0
    prev_crop_gray = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_interval != 0:
            frame_count += 1
            continue

        if saved_count >= max_frames_per_video:
            break

        face_crop = crop_largest_face(frame)

        if face_crop is None and save_center_crop_if_no_face:
            h, w = frame.shape[:2]
            side = min(h, w)
            x1 = (w - side) // 2
            y1 = (h - side) // 2
            face_crop = frame[y1:y1 + side, x1:x1 + side]

        if face_crop is None:
            frame_count += 1
            continue

        crop_gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)

        if duplicate_filter and is_near_duplicate(
            prev_crop_gray,
            crop_gray,
            diff_threshold=diff_threshold,
            change_ratio_threshold=change_ratio_threshold
        ):
            frame_count += 1
            continue

        resized = cv2.resize(face_crop, output_size, interpolation=cv2.INTER_AREA)

        frame_filename = f"{video_name}_face_{saved_count:03d}.jpg"
        frame_path = os.path.join(output_folder, frame_filename)
        cv2.imwrite(frame_path, resized)

        saved_paths.append(frame_path)
        saved_count += 1
        prev_crop_gray = crop_gray
        frame_count += 1

    cap.release()
    print(f"[{video_name}] saved {saved_count} face crops")
    return saved_paths


base_data_dir = '/Users/kunzuhaib/work/mycode/student/trimester_02/project/data'
base_out_dir = '/Users/kunzuhaib/work/mycode/student/trimester_02/project/data_optimal_frames'


for directory in ['Celeb-synthesis', 'YouTube-real', 'Celeb-real']:
    source_folder = os.path.join(base_data_dir, directory)
    label = 'real' if 'real' in directory else 'fake'

    for video in os.listdir(source_folder):
        if video.startswith('.'):
            continue

        source_video_path = os.path.join(source_folder, video)
        output_folder = get_output_folder(video, label, base_out_dir)

        extract_xception_frames(
            source_video_path=source_video_path,
            video_name=video,
            output_folder=output_folder,
            frame_interval=10,            
            max_frames_per_video=15,      
            output_size=(299, 299),       
            duplicate_filter=True,
            diff_threshold=12,
            change_ratio_threshold=0.015,
            save_center_crop_if_no_face=False
        )